# Whisper large-v3 Studio on Colab

Drop an audio or video file, get timestamped text and subtitles. This runs [openai/whisper-large-v3](https://huggingface.co/openai/whisper-large-v3) with faster-whisper on Colab's GPU.

1. **Runtime → Change runtime type → T4 GPU** (any GPU works).
2. Run the cells from top to bottom.

Code: [letsrunai/whisper-large-v3-studio](https://github.com/letsrunai/whisper-large-v3-studio)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "git+https://github.com/letsrunai/whisper-large-v3-studio#subdirectory=server"

## Open the app

This is the lite page; the full Next.js page is for running on your own machine. The first transcription downloads the weights (about 3 GB) and loads them onto the GPU, so it takes longer than the next ones.

In [ ]:
import subprocess, time, urllib.request

server = subprocess.Popen(["whisper-studio", "serve", "--port", "7860"])
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:7860")
        break
    except OSError:
        time.sleep(1)

from google.colab import output
output.serve_kernel_port_as_window(7860)

Upload a file with the folder icon on the left and put its name below. The command prints the time it took and the speed against real time; the monitor around it records the peak GPU memory.


In [ ]:
import subprocess

monitor = subprocess.Popen(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-lms", "500"],
    stdout=subprocess.PIPE, text=True,
)
!whisper-studio transcribe your-file.mp3 --srt subtitles.srt
monitor.terminate()
print("peak GPU memory:", max(int(line) for line in monitor.communicate()[0].split()), "MiB")
